In [ ]:
# Access your secret keys via
from google.colab import userdata
# The name of your secret must match `OPENAI_API_KEY`
OPENAI_API_KEY = userdata.get('OpenAI_Generic-Key')

# Import OpenAI API and set up the key
from openai import OpenAI
client = OpenAI(api_key=(OPENAI_API_KEY))

In [ ]:
import textwrap

print("Welcome to the Interactive Teacher Math Calculator!")
print("Type 'exit' or 'quit' at any prompt to stop.\n")

while True:
    try:
        n_str = input("Enter the base integer (n): ").strip()
        if n_str.lower() in ['exit', 'quit']:
            print("Goodbye!")
            break

        i_str = input("Enter the number of iterations (i): ").strip()
        if i_str.lower() in ['exit', 'quit']:
            print("Goodbye!")
            break

        n = int(n_str)
        iterations = int(i_str)

        # FIX: "Multiply n by itself i times" conceptually means n raised to the power of i.
        # If iterations is 0, result is 1. Otherwise, we multiply n by itself (iterations - 1) more times.
        if iterations == 0:
            python_result = 1
            steps = ["1"]
        else:
            python_result = n
            steps = [str(n)]
            for _ in range(iterations - 1):
                python_result *= n
                steps.append(str(n))

        expression = " * ".join(steps)
        user_question = f"Multiply {n} by itself {iterations} times. Equation: {expression} = ?"

        # First call: Ask the model to solve the math and explain
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0.7,
            max_tokens=600,
            messages=[
                {
                    "role": "developer",
                    "content": "Answer the math question asked by the user. Talk like a friendly teacher explaining each step clearly."
                    "At the very end of your response, write exactly 'FINAL_ANSWER: <number>' with the final calculation result."
                },
                {
                    "role": "user",
                    "content": f"Explain this iterative multiplication: {user_question}"
                }
            ]
        )

        output = response.choices[0].message.content

        # Display the teacher's explanation
        print("\n--- Teacher's Explanation ---\n")
        print(textwrap.fill(output, width=80))
        print("\n" + "="*40 + "\n")

        # Verification step: Compare python math with GPT response
        print(f"Python calculated result: {python_result}")

        # Try to parse the final answer from the model's text
        is_correct = False
        gpt_result = None
        if "FINAL_ANSWER:" in output:
            gpt_num_part = output.split("FINAL_ANSWER:")[-1].strip().split()[0].replace(",", "").strip(".").strip()
            try:
                gpt_result = int(gpt_num_part)
                if python_result == gpt_result:
                    is_correct = True
                    print("✅ VERIFICATION SUCCESSFUL: GPT-4o-mini's math matches Python exactly!")
                else:
                    print(f"❌ VERIFICATION FAILED: GPT-4o-mini calculated {gpt_result}, but Python calculated {python_result}.")
            except ValueError:
                print("⚠️ Could not parse the final numeric answer from the model's output for comparison.")
        else:
            if str(python_result) in output:
                is_correct = True
                print("✅ VERIFICATION SUCCESSFUL: The Python result was found within the teacher's explanation.")
            else:
                print("❌ VERIFICATION FAILED: The correct Python result was not found in the teacher's response.")

        # Dynamic Reaction if verification fails:
        if not is_correct:
            print("\n🤖 Generating self-deprecating joke from the teacher...")
            joke_response = client.chat.completions.create(
                model="gpt-4o-mini",
                temperature=1.0,
                messages=[
                    {
                        "role": "developer",
                        "content": "You are a friendly math teacher who just realized they made a mistake in their manual calculations. Write a short, humorous, self-deprecating response acknowledging that Python proved you wrong."
                    },
                    {
                        "role": "user",
                        "content": f"I calculated {gpt_result if gpt_result is not None else 'something else'}, but Python calculated {python_result}. Give me a funny excuse or a self-deprecating joke!"
                    }
                ]
            )
            print(f"\nTeacher: \"{joke_response.choices[0].message.content.strip()}\"\n")

        print("\n" + "-"*60 + "\nReady for the next calculation!\n")

    except ValueError:
        print("❌ Error: Please enter valid integers for both fields.\n")
    except Exception as e:
        print(f"An error occurred: {e}\n")
        break

Welcome to the Interactive Teacher Math Calculator!
Type 'exit' or 'quit' at any prompt to stop.

Enter the base integer (n): 7
Enter the number of iterations (i): 20

--- Teacher's Explanation ---

Absolutely! What you're looking at is a way to express the multiplication of a
number by itself multiple times. When we multiply a number by itself repeatedly,
we can use exponents to make it simpler to write and understand.  In your case,
multiplying 7 by itself 20 times can be expressed as \( 7^{20} \). Here’s how we
can break it down step by step:  1. **Understanding Exponents**: The expression
\( 7^{20} \) means that we take the base number, which is 7, and multiply it by
itself a total of 20 times.  2. **Using a Calculator**: To calculate \( 7^{20}
\), it's usually best to use a calculator due to the large size of the number.
If you were to do it by hand, it would involve a lot of multiplication!  3.
**Performing the Calculation**: When you enter \( 7^{20} \) into a calculator,
it will

KeyboardInterrupt: Interrupted by user